In [1]:
doc = """
It can still answer this through co-occurrence patterns, Alice and Company X appearing in the same sentences about employment, but a well-constructed knowledge graph with accurate relations would be more precise for this class of query.

The operative phrase is “well-constructed”. The GraphRAG-Bench results show that most real-world knowledge graphs built by LLMs contain enough noisy relations that explicit modeling hurts more than it helps.

LinearRAG’s bet is that avoiding noise entirely outweighs the loss of expressiveness, and the benchmark results support that bet across all datasets tested.

"""

In [17]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def build_passages(doc, chunk_size=5):
    """
    Split a document into fixed-size sentence chunks (passages).

    This function tokenizes the input document into sentences and groups them
    into passages of a specified size. Each passage contains a list of sentences
    and a unique passage ID.

    Args:
        doc (str): Input text document to be split into passages.
        chunk_size (int, optional): Number of sentences per passage.
            Defaults to 5.

    Returns:
        List[dict]: A list of passages, where each passage is a dictionary with:
            - "passage_id" (str): Unique identifier for the passage (e.g., "P0", "P5", ...).
            - "sentences" (List[str]): List of sentences in the passage.

    Example:
        >>> doc = "Sentence one. Sentence two. Sentence three. Sentence four."
        >>> build_passages(doc, chunk_size=2)
        [
            {"passage_id": "P0", "sentences": ["Sentence one.", "Sentence two."]},
            {"passage_id": "P2", "sentences": ["Sentence three.", "Sentence four."]}
        ]

    Notes:
        - Sentence tokenization depends on the behavior of `sent_tokenize`.
        - The last passage may contain fewer sentences if the total number
          of sentences is not divisible by `chunk_size`.
    """
    sentences = sent_tokenize(doc)
    print(f">>> cnt of sentences: {len(sentences)}")
    passages = []

    for i in range(0, len(sentences), chunk_size):
        chunk = sentences[i:i+chunk_size]
        passages.append({
            "passage_id": f"P{i}",
            "sentences": chunk
        })
    return passages

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jongb\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [39]:
passages = build_passages(doc=doc, chunk_size=3)
passages

>>> cnt of sentences: 4


[{'passage_id': 'P0',
  'sentences': ['\nIt can still answer this through co-occurrence patterns, Alice and Company X appearing in the same sentences about employment, but a well-constructed knowledge graph with accurate relations would be more precise for this class of query.',
   'The operative phrase is “well-constructed”.',
   'The GraphRAG-Bench results show that most real-world knowledge graphs built by LLMs contain enough noisy relations that explicit modeling hurts more than it helps.']},
 {'passage_id': 'P3',
  'sentences': ['LinearRAG’s bet is that avoiding noise entirely outweighs the loss of expressiveness, and the benchmark results support that bet across all datasets tested.']}]

In [71]:
import spacy

nlp = spacy.load("../model/en_core_web_sm")

def extract_entities(text):
    doc = nlp(text)
    return list(set([ent.text for ent in doc.ents]))
    # return list(set([ent.text for ent in doc if ent.pos_=="NOUN"]))  # 명사 추출 방식

d:\auto_vectordb\.venv\Lib\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [72]:
text = "When to use Graphs in RAG: A Comprehensive Benchmark and Analysis for Graph Retrieval-Augmented Generation"
ner = extract_entities(text=text)
ner

['Graphs', 'Analysis for Graph Retrieval-Augmented Generation']

In [73]:
## Domain NER 커스터마이징
import spacy

nlp = spacy.load("../model/en_core_web_sm")

ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "PRODUCT", "pattern": "GPT-4"},
    {"label": "PRODUCT", "pattern": "GRAPH"},
    {"label": "ERROR", "pattern": [{"TEXT": {"REGEX": "ERR[0-9]+"}}]},
    {"label": "API", "pattern": "OpenAI API"}
]

ruler.add_patterns(patterns)
doc = nlp("ERR1123 occurred in GPT-4 using OpenAI API")
[(ent.text, ent.label_) for ent in doc.ents]

[('ERR1123', 'ERROR'), ('GPT-4', 'PRODUCT'), ('OpenAI API', 'API')]

In [74]:
doc = nlp(text.upper())
[(ent.text, ent.label_) for ent in doc.ents]

[('ANALYSIS FOR', 'WORK_OF_ART'), ('GRAPH', 'PRODUCT')]

In [75]:
def build_graph(passages):
    sentence_nodes = []
    entity_map = {}

    for p in passages:
        for i, sent in enumerate(p["sentences"]):
            sid = f"S_{i}"
            entities = extract_entities(sent)

            sentence_nodes.append({
                "sentence_id": sid,
                "text": sent,
                "entities": entities,
                "passage_id": p["passage_id"]
            })

            for e in entities:
                entity_map.setdefault(e, {
                    "entity": e,
                    "sentences": [],
                    "passages": set()
                })
                entity_map[e]["sentences"].append(sid)
                entity_map[e]["passages"].add(p["passage_id"])

    return sentence_nodes, list(entity_map.values())

In [76]:
res = build_graph(passages=passages)
res

([{'sentence_id': 'S_0',
   'text': '\nIt can still answer this through co-occurrence patterns, Alice and Company X appearing in the same sentences about employment, but a well-constructed knowledge graph with accurate relations would be more precise for this class of query.',
   'entities': ['Alice'],
   'passage_id': 'P0'},
  {'sentence_id': 'S_1',
   'text': 'The operative phrase is “well-constructed”.',
   'entities': [],
   'passage_id': 'P0'},
  {'sentence_id': 'S_2',
   'text': 'The GraphRAG-Bench results show that most real-world knowledge graphs built by LLMs contain enough noisy relations that explicit modeling hurts more than it helps.',
   'entities': ['The GraphRAG-Bench'],
   'passage_id': 'P0'},
  {'sentence_id': 'S_0',
   'text': 'LinearRAG’s bet is that avoiding noise entirely outweighs the loss of expressiveness, and the benchmark results support that bet across all datasets tested.',
   'entities': ['LinearRAG'],
   'passage_id': 'P3'}],
 [{'entity': 'Alice', 'senten